# Baseline - longest option

Ankur 21f2000153

No model. Just pick the longest option.

This is the score the real models have to beat.

## Load data

In [ ]:
import re
import numpy as np
import pandas as pd

DATA_FOLDER = "/kaggle/input/competitions/smart-mcq-solver-challenge/"
OPTIONS = ["A", "B", "C", "D", "E"]
LETTER_TO_INDEX = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
SEED = 42
VALIDATION_FRACTION = 0.20

np.random.seed(SEED)

train_df = pd.read_csv(DATA_FOLDER + "train.csv")
test_df = pd.read_csv(DATA_FOLDER + "test.csv")

print("train rows:", len(train_df))
print("test rows :", len(test_df))

In [ ]:
PREFIXES = [
    "Pick the best possible answer:",
    "Select the most accurate option:",
    "Identify the correct statement:",
    "Determine the correct option:",
    "Choose the correct answer:",
    "Which of the following is correct?",
]

SUFFIXES = [
    "among the listed options.",
    "from the following choices.",
    "carefully.",
    "based on the given context.",
    "among the list of options.",
]


def clean_prompt(text):
    text = str(text).strip()

    for prefix in PREFIXES:
        if text.startswith(prefix):
            text = text[len(prefix):]
            text = text.strip()
            break

    for suffix in SUFFIXES:
        if text.endswith(suffix):
            text = text[:-len(suffix)]
            text = text.strip()
            break

    return text


train_df["question"] = train_df["prompt"].apply(clean_prompt)
test_df["question"] = test_df["prompt"].apply(clean_prompt)

y_all = train_df["answer"].map(LETTER_TO_INDEX).values

print("before:", train_df["prompt"].iloc[0][:90])
print("after :", train_df["question"].iloc[0][:90])

## Split

Grouped so the same question does not appear on both sides.

In [ ]:
def normalize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return text.strip()


def make_question_key(row):
    option_texts = []
    for letter in OPTIONS:
        option_texts.append(normalize(row[letter]))
    option_texts.sort()
    return "|".join(option_texts)


train_df["key"] = train_df.apply(make_question_key, axis=1)

unique_keys = train_df["key"].unique().tolist()
shuffler = np.random.RandomState(SEED)
shuffler.shuffle(unique_keys)

number_of_validation_keys = int(VALIDATION_FRACTION * len(unique_keys))
validation_keys = set(unique_keys[:number_of_validation_keys])

is_validation = train_df["key"].isin(validation_keys).values

val_df = train_df[is_validation].reset_index(drop=True)
fit_df = train_df[~is_validation].reset_index(drop=True)

y_val = val_df["answer"].map(LETTER_TO_INDEX).values
y_fit = fit_df["answer"].map(LETTER_TO_INDEX).values

shared = set(val_df["key"]) & set(fit_df["key"])

print("total rows       :", len(train_df))
print("unique questions :", train_df["key"].nunique())
print("rows for fitting :", len(fit_df))
print("rows for testing :", len(val_df))
print("shared questions :", len(shared))

## MAP@3

In [ ]:
def map3(scores, labels):
    total = 0.0

    for i in range(len(labels)):
        current_row = scores[i]
        ranking = sorted(range(5), key=lambda j: current_row[j], reverse=True)
        position = ranking.index(labels[i])

        if position < 3:
            total = total + 1 / (position + 1)

    return total / len(labels)


def accuracy(scores, labels):
    best_option = scores.argmax(axis=1)
    return float((best_option == labels).mean())


def macro_f1(scores, labels):
    from sklearn.metrics import f1_score
    best_option = scores.argmax(axis=1)
    return float(f1_score(labels, best_option, average="macro",
                          labels=[0, 1, 2, 3, 4], zero_division=0))


def evaluate(scores, labels, name):
    results = {
        "val_accuracy": accuracy(scores, labels),
        "val_macro_f1": macro_f1(scores, labels),
        "val_map3": float(map3(scores, labels)),
    }
    print(name)
    print("   accuracy :", round(results["val_accuracy"], 4))
    print("   macro F1 :", round(results["val_macro_f1"], 4))
    print("   MAP@3    :", round(results["val_map3"], 4))
    return results

In [ ]:
perfect_scores = np.array([[9, 0, 0, 0, 0], [0, 9, 0, 0, 0]])
second_place = np.array([[0, 9, 0, 0, 0], [9, 0, 0, 0, 0]])
labels = np.array([0, 1])

print("answer ranked first  :", map3(perfect_scores, labels))
print("answer ranked second :", map3(second_place, labels))
print("uniform random guess :", round((1 + 1/2 + 1/3) / 5, 4))

## Baseline

In [ ]:
def length_scores(dataframe):
    columns = []
    for letter in OPTIONS:
        lengths = dataframe[letter].astype(str).str.len()
        columns.append(lengths)
    return np.column_stack(columns).astype(float)


baseline_scores = length_scores(val_df)
baseline_results = evaluate(baseline_scores, y_val, "LENGTH BASELINE")

## Predictions

In [ ]:
test_scores = length_scores(test_df)
top_three = np.argsort(-test_scores, axis=1)[:, :3]

predictions = []
for row in top_three:
    letters = []
    for option_number in row:
        letters.append(OPTIONS[option_number])
    predictions.append(" ".join(letters))

submission = pd.DataFrame({"ID": test_df["id"], "Prediction": predictions})
submission.to_csv("submission_baseline.csv", index=False)

print(submission.head())
print("rows:", len(submission))

## Summary

Sorting by length beats random guessing without reading the question. The wrong options are shorter because they were made by rewriting the correct answer.